# K and LS Layer
This notebook is to create both layers based on Claude's recommendations.

## Claude's recommendations
### Methods — K (soil erodibility)
| # | Method | Paper | Formula | Inputs | Effort |
|---|---|---|---|---|---|
| K1 | **Wischmeier & Smith (1978) nomograph** + stone correction | Borrelli et al. 2022 (GloSEM; also current v1.1 layer) | $K=\frac{2.1\cdot10^{-4}M^{1.14}(12-OM)+3.25(s-2)+2.5(p-3)}{100}\cdot0.1317$, with $M=(\%silt+\%vfs)(100-\%clay)$; $K_{st}=K\cdot e^{-0.04(R_c-10)}$ for $R_c>10\%$ | Sand, silt, clay, OM, coarse fragments (SoilGrids); structure class $s$ and permeability class $p$ from texture-class lookups; very fine sand ($vfs$) estimated (not in SoilGrids) | Medium (lookups and a vfs assumption) |
| K2 | **Williams / EPIC** (Sharpley & Williams 1990) | Pyne et al. 2026 | $K=[0.2+0.3e^{-0.0256Sd(1-St/100)}]\,[\frac{St}{Cy+St}]^{0.3}\,[1-\frac{0.25OC}{OC+e^{3.72-2.95OC}}]\,[1-\frac{0.7SN}{SN+e^{-5.51+22.9SN}}]\times0.1317$ | Sand, silt, clay, OC (SoilGrids) | **Low**, pure raster algebra |
| K3 | **Renard et al. (1997) geometric mean diameter** | Fadl et al. 2025 | $K=0.0034+0.0405\exp[-0.5(\frac{\log D_g+1.659}{0.7101})^2]$ | Sand, silt, clay | Very low, but ignores OM and stones |

### Methods — LS (slope length and steepness)
| # | Method | Paper | Formula / approach | Inputs | Effort |
|---|---|---|---|---|---|
| LS1 | **Desmet & Govers (1996)** 2-D unit contributing area | Borrelli et al. 2022 | $L_{i,j}=\frac{(A_{i,j}+D^2)^{m+1}-A_{i,j}^{m+1}}{D^{m+2}\,x_{i,j}^m\,22.13^m}$ with $m=\beta/(1+\beta)$, $\beta=\frac{\sin\theta/0.0896}{3\sin^{0.8}\theta+0.56}$; S from McCool (1987) | 90 m DEM and upslope contributing area (SRTM/ASTER in GloSEM; SAGA, 870 tiles) | High if flow routing is done in-house. **Medium if precomputed flow accumulation is used** (MERIT Hydro) |
| LS2 | **Slope-class exponent + McCool S** (Liu steep-slope variant) | Pyne et al. 2026 | $L=(\lambda/22.13)^m$, $m$ = 0.2/0.3/0.4/0.5 by slope class; S = 10.8 sinθ + 0.03 (< 9%), 16.8 sinθ − 0.5 (9–18%), 21.9 sinθ − 0.96 (≥ 18%) | DEM, flow length | Medium |
| LS3 | **Shinde et al. (2011)** | Fadl et al. 2025 | $LS=(L/22.1)^m(65.4\sin^2\theta+4.56\sin\theta+0.0065)$ | DEM, slope length | Low, but uncapped lengths give unrealistic LS (up to 4,110 in the paper) |

### Selected K and LS methods
- **K → K1 (GloSEM nomograph + stone correction), with K2 (EPIC) computed alongside as a check.**
  - K1 is what GloSEM 1.1/1.3 and the current $K_{st}$ layer use, so the new layer stays comparable with the existing LEAFs.
  - K2 is almost free once the SoilGrids layers are downloaded. Where the two disagree strongly, the nomograph assumptions (vfs, $s$, $p$) can be flagged.
  - If the vfs and structure lookups cannot be reproduced defensibly, **fall back to K2**. It is the easiest method fully supported by open data.
- **LS → LS1 (Desmet & Govers, as GloSEM), using precomputed MERIT Hydro flow accumulation and MERIT DEM slope at 3″ (~90 m), aggregated to 5′.**
  - This reproduces the GloSEM method without running global flow routing.
  - It can run in Google Earth Engine, which hosts MERIT DEM and MERIT Hydro, or locally tile by tile.

### Method breakdown
#### K Layer
1. **Download SoilGrids 2.0** mean layers for **0–5, 5–15 and 15–30 cm**: `sand`, `silt`, `clay` (g/kg), `soc` (dg/kg), `cfvo` (cm³/dm³). Use the 1 km EPSG:4326 aggregated release if available, otherwise the 250 m VRTs. Depth-weight them to 0–30 cm, consistent with SOC.
2. Convert units:
   - OM % = 1.724 × SOC %, with SOC % = soc / 100.
   - Coarse fragments $R_c$ % = cfvo / 10.
3. Classify the USDA texture class per pixel, then:
   - **permeability $p$**: sand = 1; loamy sand, sandy loam = 2; loam, silt loam, silt = 3; sandy clay loam, clay loam = 4; silty clay loam, sandy clay = 5; silty clay, clay = 6 (Panagos et al. 2014 / USDA).
   - **structure $s$**: lookup by texture and OM class, following Borrelli et al. 2017 SI / Panagos et al. 2014.
   - **very fine sand**: estimated as a fixed fraction of sand. This assumption must be documented and tested for sensitivity. Note that the nomograph is valid for silt + vfs ≤ 70%.
4. Compute $K$ (nomograph) and $K_{st}=K\cdot e^{-0.04(R_c-10)}$ (for $R_c>10\%$) at native resolution. Compute $K_{EPIC}$ in parallel.
5. Aggregate both to 5′ by mean.

Output files:
- `rusle_Kst_5m.tif`
- `rusle_K_epic_5m.tif` (QA)

#### LS Layer
1. **Inputs:** MERIT DEM (slope θ) and MERIT Hydro `upa` (upstream drainage area), both at 3″. Coverage runs from 60°S to 90°N, which spans the UHTH crop extent.
2. **Contributing area per cell:** $A = upa\cdot10^6 - D^2$ (m²).
   - **Cap A** to a maximum hillslope length (default $\lambda_{max}$ = 300 m, i.e. $A\le\lambda_{max}\cdot D$). Run a sensitivity test at 100–500 m.
   - Mask channel cells (upa above a threshold). This avoids the unrealistic LS seen in Fadl et al. (2025), and represents field-scale slope lengths.
3. **L:** Desmet & Govers with McCool β and $x = |\sin\alpha|+|\cos\alpha|$ (aspect α). **S:** McCool (1987), $S=10.8\sin\theta+0.03$ for tan θ < 0.09 and $S=16.8\sin\theta-0.50$ otherwise (Nearing 1997 as an option for very steep slopes).
4. **Aggregate to 5′ twice:**
   - **`LS_all`**: mean over all land. Used for forest, grassland and urban LEAFs.
   - **`LS_crop`**: cropland-weighted mean. Weights come from ESA WorldCover cropland fraction, or `ESA100_Cropland_fraction.tif` from GloSEM 1.3. Crops sit on flatter ground than the cell average, so crop LEAFs should use `LS_crop`.
5. **Process 5° tiles** (or run in Earth Engine) and write only the 5′ aggregates. Never hold a global 90 m LS raster in memory.

Output files:
- `rusle_LS_all_5m.tif`
- `rusle_LS_crop_5m.tif`

### Validation of the new layers
- Aggregate the new R, $K_{st}$ and LS to the GloSEM v1.1 0.2083° grid. Compare with `RUSLE_{R,K,LS}Factor_v1.1_25km.tif`: correlation, bias map, and ratio by continent and biome.
- Compare $K$ (nomograph) with $K_{EPIC}$.
- Compare the MFI-ratio scenarios with the published CMIP5 future-R ratios (Panagos et al. 2022). Their global average change is +26–29% by 2050.
- Re-run a handful of LEAF flows (wheat, maize, soy, coffee, grassland) with the new RLSK. Compare country, subcountry and ecoregion means with the current `se_*_clipped_2` outputs, and with GloSEM 1.3 cropland totals (12.3 Mg ha⁻¹ yr⁻¹ global mean in 2019).
- Re-check whether the current outlier clipping (log1p winsor 0.5–99.5) is still needed at 5′.


# K Creation

## EPIC - K
Due to data limitations on permeability and soil structure class, EPIC-K calculation is implemented for the basis. It has been implemented in the sbtn_leaf.rusle module, with the original and Yang et al. (2024) corrections.

### Running calculations

Importing modules

In [1]:
import sbtn_leaf.rusle as rusle
import sbtn_leaf.map_plotting as mp
import sbtn_leaf.map_calculations as mc
import importlib 

Could not determine dtype for column 1, falling back to string


Running calculations

In [2]:
k_epic_fp = '../../RUSLE/outputs/k_epic.tif'
k_epic_corr_fp = '../../RUSLE/outputs/k_epic_corr.tif'

In [9]:
importlib.reload(rusle)

<module 'sbtn_leaf.rusle' from 'C:\\Users\\loyola\\OneDrive - World Wildlife Fund, Inc\\Documents\\203. Python projects\\SBTN_Test\\src\\sbtn_leaf\\rusle.py'>

In [3]:
rusle.calculate_K_Epic(k_epic_fp, k_epic_corr_fp)

Running K EPIC calculations...
     Loading data
     Calculating K Epic
     Writing...
     K EPIC saved into ../../RUSLE/outputs/k_epic.tif
     K EPIC corrected saved into ../../RUSLE/outputs/k_epic_corr.tif
All done!


### Comparison with original Borelli
K Calculated, now analyzing and comparing...

In [4]:
k_og_fp = "../../data/soil_erosion/GloSEM_v1.1/RUSLE_KFactor_v1.1_25km.tif"

## Structure (s), permeability (p) and very fine sand (vfs)
Built from the SoilGrids 2.0 1 km layers in `data/soilgrids_1000m` (Homolosine, ESRI:54052), depth-weighted to 0–30 cm (5/10/15 cm). They stay on the native grid, because K is non-linear and is computed before aggregating to 5′. Method sources: [papers_summary.md §4–5](../papers/papers_summary.md) (Panagos et al. 2014; AH 537).
- **Texture**: USDA triangle, after rescaling sand + silt + clay to 100 %.
- **p**: texture lookup (`rusle.PERMEABILITY_BY_TEXTURE`, Panagos et al. 2014 Table 2): sand 1; loamy sand, sandy loam 2; loam, silt loam, silt 3; sandy clay loam, clay loam 4; silty clay loam, sandy clay 5; silty clay, clay 6. It is then calibrated with coarse fragments (`cfvo`): $K_b = K_f(1-R_w)$ with Rawls et al. (1982) $K_f$, and p is reclassified (`use_coarse_fragments=True`).
- **s**: texture lookup (`rusle.STRUCTURE_BY_TEXTURE`, our assumption, because Panagos used the EU-only ESDB): sand, loamy sand 1; sandy loam, loam, silt loam, silt 2; sandy clay loam, clay loam, silty clay loam 3; sandy clay, silty clay, clay 4. Use `structure_method="constant"` for s = 2 everywhere. An OM adjustment (`om_thresholds`) is optional and meant for sensitivity tests.
- **vfs**: 20 % of sand (Panagos et al. 2014). Change it with `vfs_fraction` to test sensitivity.

Outputs go to `data/soil_erosion/K_1km/`.

In [ ]:
from sbtn_leaf import rusle

# Needs sand/silt/clay/cfvo for 0-5, 5-15 and 15-30 cm to be fully downloaded (~3 min)
k_inputs = rusle.build_structure_permeability_vfs()
k_inputs

## RUSLE2-K (Yang et al. 2024)
Nomograph with the RUSLE2 vfs estimate, $M_{vfs}=(0.74-0.62\,sand/100)\,sand$, and a flattened texture term above 68 % silt + vfs. OM is capped at 4 %. It uses the s and p rasters built above. Two layers are written:
- `K_rusle2_0-30cm.tif`: uncorrected, comparable with GloSEM / Panagos K.
- `K_rusle2_yang_corrected_0-30cm.tif`: with the Yang et al. measured-K correction $2.1311K-0.0359$, floored at 0.

See [papers_summary.md §6](../papers/papers_summary.md) for caveats.

In [ ]:
# Needs sand/silt/clay/soc for all three depths, and the s/p rasters from the cell above (~3 min)
k_rusle2 = rusle.build_k_rusle2()
k_rusle2